In [0]:
spark.sql("""
CREATE OR REPLACE VIEW ecommerce_returns.gold.vw_business_kpis AS

SELECT
    o.total_orders,
    r.total_returns,
    r.return_rate,
    f.total_refunds,
    f.total_refund_amount,
    f.avg_refund_processing_days,
    f.refund_sla_breaches,
    f.refund_sla_breach_rate,
    f.refund_rate

FROM
(
    SELECT
        COUNT(DISTINCT order_id) AS total_orders
    FROM ecommerce_returns.gold.fact_orders
) o

CROSS JOIN

(
    SELECT
        COUNT(DISTINCT return_id) AS total_returns,
        COUNT(DISTINCT order_id) * 100.0 /
            (SELECT COUNT(DISTINCT order_id)
             FROM ecommerce_returns.gold.fact_orders) AS return_rate
    FROM ecommerce_returns.gold.fact_returns
) r

CROSS JOIN

(
    SELECT
        COUNT(DISTINCT refund_id) AS total_refunds,
        SUM(refund_amount) AS total_refund_amount,
        AVG(refund_processing_days) AS avg_refund_processing_days,
        SUM(CASE WHEN sla_breached = true THEN 1 ELSE 0 END)
            AS refund_sla_breaches,
        SUM(CASE WHEN sla_breached = true THEN 1 ELSE 0 END) * 100.0
            / COUNT(DISTINCT refund_id) AS refund_sla_breach_rate,
        COUNT(DISTINCT return_id) * 100.0 /
            (SELECT COUNT(DISTINCT return_id)
             FROM ecommerce_returns.gold.fact_returns) AS refund_rate
    FROM ecommerce_returns.gold.fact_refunds
) f
""")

display(spark.sql("""
SELECT *
FROM ecommerce_returns.gold.vw_business_kpis
"""))

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW ecommerce_returns.gold.vw_top_returned_products AS
SELECT
    r.product_id,
    o.category,
    o.subcategory,
    COUNT(DISTINCT r.return_id) AS total_returns,
    SUM(r.return_quantity) AS returned_quantity,
    COUNT(DISTINCT r.order_id) AS returned_orders
FROM ecommerce_returns.gold.fact_returns r
JOIN ecommerce_returns.gold.fact_orders o
    ON r.order_id = o.order_id
GROUP BY
    r.product_id,
    o.category,
    o.subcategory
""")

display(spark.sql("""
SELECT *
FROM ecommerce_returns.gold.vw_top_returned_products
ORDER BY total_returns DESC
LIMIT 20
"""))

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW ecommerce_returns.gold.vw_top_returned_categories AS
SELECT
    o.category,
    COUNT(DISTINCT r.return_id) AS total_returns,
    SUM(r.return_quantity) AS returned_quantity,
    COUNT(DISTINCT r.order_id) AS returned_orders
FROM ecommerce_returns.gold.fact_returns r
JOIN ecommerce_returns.gold.fact_orders o
    ON r.order_id = o.order_id
GROUP BY o.category
""")

display(spark.sql("""
SELECT *
FROM ecommerce_returns.gold.vw_top_returned_categories
ORDER BY total_returns DESC
"""))

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW ecommerce_returns.gold.vw_seller_return_rates AS
SELECT
    o.seller_id,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT r.order_id) AS returned_orders,
    COUNT(DISTINCT r.return_id) AS total_returns,
    COUNT(DISTINCT r.order_id) * 100.0
        / COUNT(DISTINCT o.order_id) AS return_rate
FROM ecommerce_returns.gold.fact_orders o
LEFT JOIN ecommerce_returns.gold.fact_returns r
    ON o.order_id = r.order_id
GROUP BY o.seller_id
""")

display(spark.sql("""
SELECT *
FROM ecommerce_returns.gold.vw_seller_return_rates
ORDER BY return_rate DESC
"""))

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW ecommerce_returns.gold.vw_monthly_return_trend AS
SELECT
    DATE_TRUNC('month', r.return_date) AS return_month,
    COUNT(DISTINCT r.return_id) AS total_returns,
    SUM(r.return_quantity) AS returned_quantity,
    COUNT(DISTINCT r.order_id) AS returned_orders
FROM ecommerce_returns.gold.fact_returns r
GROUP BY DATE_TRUNC('month', r.return_date)
""")

display(spark.sql("""
SELECT *
FROM ecommerce_returns.gold.vw_monthly_return_trend
ORDER BY return_month
"""))

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW ecommerce_returns.gold.vw_return_reasons AS
SELECT
    return_reason,
    COUNT(DISTINCT return_id) AS total_returns,
    SUM(return_quantity) AS returned_quantity
FROM ecommerce_returns.gold.fact_returns
GROUP BY return_reason
""")

display(spark.sql("""
SELECT *
FROM ecommerce_returns.gold.vw_return_reasons
ORDER BY total_returns DESC
"""))

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW ecommerce_returns.gold.vw_refund_metrics AS
SELECT
    COUNT(DISTINCT refund_id) AS total_refunds,
    SUM(refund_amount) AS total_refund_amount,
    AVG(refund_amount) AS average_refund_amount,
    AVG(refund_processing_days) AS average_processing_days,
    SUM(CASE WHEN sla_breached = true THEN 1 ELSE 0 END) AS sla_breaches,
    SUM(CASE WHEN sla_breached = true THEN 1 ELSE 0 END) * 100.0
        / COUNT(DISTINCT refund_id) AS sla_breach_rate
FROM ecommerce_returns.gold.fact_refunds
""")

display(spark.sql("""
SELECT *
FROM ecommerce_returns.gold.vw_refund_metrics
"""))